<a href="https://colab.research.google.com/github/fphsFischmeister/ILAE_NeuroimagingSchool/blob/master/notebooks/06_connectivity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Google Colab"/>  </a>

# Welcome to the interactive ILAE workshop on task-based activation detection.

Author: Florian Ph.S Fischmeister, Marc Berger, Radeshyam Stepponat

---
- Part 1: Preprocessing fMRI data with fMRIPrep [Jupyter Notebook](https://colab.research.google.com/github/fphsFischmeister/ILAE_NeuroimagingSchool/blob/master/notebooks/01_preprocessing.ipynb)
- Part 2: First Level of a simple motor task [Jupyter Notebook](https://colab.research.google.com/github/fphsFischmeister/ILAE_NeuroimagingSchool/blob/master/notebooks/02_basic_motor_task.ipynb)
- Part 3: A Home-Town-Walking language paradigm [Jupyter Notebook](https://colab.research.google.com/github/fphsFischmeister/ILAE_NeuroimagingSchool/blob/master/notebooks/03_hometown_task.ipynb)
- Part 4: Phrases, a language paradigm [Jupyter Notebook](https://colab.research.google.com/github/fphsFischmeister/ILAE_NeuroimagingSchool/blob/master/notebooks/04_phases_task.ipynb)
- Part 5: All language tasks [Jupyter Notebook](https://colab.research.google.com/github/fphsFischmeister/ILAE_NeuroimagingSchool/blob/master/notebooks/05_all_language_task.ipynb)
- Part 6: Functional Connectivity [Jupyter Notebook](https://colab.research.google.com/github/fphsFischmeister/ILAE_NeuroimagingSchool/blob/master/notebooks/06_connectivity.ipynb)



## 1. Prepare the notebook


In [ ]:
# get some data for presentation
!rm -rf ILAE_NeuroimagingSchool
!git clone https://github.com/fphsFischmeister/ILAE_NeuroimagingSchool.git

# get some functional motor data
!wget https://dinlab.roentgen.meduniwien.ac.at/ILAE_NeuroimagingSchool/dataset/sub-ILAEDemo001_ses-01_task-Rest_run-01_space-MNI_desc-denoised_bold.nii.gz -P ILAE_NeuroimagingSchool/dataset/func/


In [ ]:
# install modules
%pip install nilearn
%pip install ipyniivue
%pip install ipywidgets
%pip install matplotlib

import matplotlib.pyplot as plt
import nibabel as nib
import nilearn
import numpy as np
import pandas as pd
from nilearn import datasets, image, plotting


In [ ]:
# some basic analysis definitions for later use
task_label = "Rest"

nifti_filename = "./ILAE_NeuroimagingSchool/dataset/func/sub-ILAEDemo001_ses-01_task-Rest_run-01_space-MNI_desc-denoised_bold.nii.gz"
nifti_filename_new = "./ILAE_NeuroimagingSchool/outputs/sub-ILAEDemo001_ses-01_task-Rest_run-01_space-MNI_desc-noNAN_bold.nii.gz"

confounds_filename = "./ILAE_NeuroimagingSchool/dataset/func/sub-ILAEDemo001_ses-01_task-Rest_run-01_desc-confounds_timeseries.tsv"

output_dir = Path("./ILAE_NeuroimagingSchool/outputs/")
output_dir.mkdir(parents=True, exist_ok=True)



In [ ]:
# Replace NaNs and infinities with 0 so Nilearn can process the image.
img = nib.load(nifti_filename)
data = img.get_fdata()
data_no_nan = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
new_img = nib.Nifti1Image(data_no_nan, img.affine, img.header)
nib.save(new_img, nifti_filename_new)

print(f"Saved cleaned image to: {nifti_filename_new}")
print(f"Image shape: {new_img.shape}")

# Part 6: Connectome of a simple Resting-State

This notebook is intended for the functional-connectivity part of the workshop. It is designed to follow a short XCP-D demonstration and then make the main analysis ideas visible in Python: extract regional BOLD time series, compute ROI-to-ROI correlations, visualize a connectome, and discuss what these quantities mean in a language-mapping context.

XCP-D is a postprocessing pipeline that starts from fMRIPrep-style derivatives and produces denoised BOLD images, parcellated time series, functional-connectivity matrices, and quality-control reports. In a production workflow, you would usually use the XCP-D outputs directly, especially the parcellated time-series files and Pearson-correlation relationship matrices.

task and the goal is static residual connectivity, task regressors may need to be handled explicitly; otherwise, the task structure can dominate the correlation matrix.

Useful references for this section:

- XCP-D documentation: <https://xcp-d.readthedocs.io/en/latest/>
- XCP-D citation: Mehta, K., Salo, T., Madison, T. J., Adebimpe, A., Bassett, D. S., Bertolero, M., … & Satterthwaite, T. D. (2024). XCP-D: A Robust Pipeline for the post-processing of fMRI data. Imaging Neuroscience, 2, 1-26. doi:10.1162/imag_a_00257.

Parts of the notebook originate from the great work of Peer Herholz and can be found at https://peerherholz.github.io/workshop_weizmann/advanced/functional_connectivity.html


## XCP-D Worklow

![XCP-D workflow](./ILAE_NeuroimagingSchool/notebooks/xcp_figure_1.png)

## Visual quality check
Before computing connectivity, always look at the data and the output of XCP-D. Connectivity estimates are sensitive to artifacts, coverage, registration, and motion. 

The full output of XCP-D can be found at: https://dinlab.roentgen.meduniwien.ac.at/ILAE_NeuroimagingSchool/xcpd



## Brain parcellation
A functional connectome needs regions we want to extract the signal from (nodes). One common way to define nodes is a parcellation atlas, where each voxel belongs to a labeled region. Here we use the Harvard-Oxford cortical atlas as a simple didactic example.

For language applications, this atlas is convenient but coarse. It contains regions such as inferior frontal, temporal, and angular gyri, but it is not a patient-specific language atlas. In clinical teaching, emphasize that atlas labels should not be treated as individual functional localization.

In [ ]:
# Download Harvard-Oxford cortical atlas.
atlas_ho = datasets.fetch_atlas_harvard_oxford("cort-maxprob-thr25-2mm")

# Location of the atlas image and labels for each region.
atlas_file = atlas_ho.maps
labels = atlas_ho.labels[1:]  # drop background label

print(f"Number of atlas regions: {len(labels)}")
print(labels[:10])

# Visualize the parcellation.
plotting.plot_roi(atlas_file, draw_cross=False, annotate=False, title="Harvard-Oxford cortical atlas")


Extract one regional time series per atlas parcelExtract timeseries to construct a functional connectome

Here we average the BOLD signal within each labeled atlas region. The result is a two-dimensional array with shape `n_timepoints × n_regions`. Each column is the mean BOLD signal from one region.

If the input is **fMRIPrep preprocessed BOLD**, pass confounds to the masker or use a dedicated postprocessing workflow. If the input is **XCP-D denoised BOLD**, do not automatically regress the same confounds again. The code below defaults to `confounds=None` because the demonstration image is already denoised.

In [ ]:
try:
    from nilearn.maskers import NiftiLabelsMasker
except ImportError:  # older Nilearn versions
    from nilearn.input_data import NiftiLabelsMasker

use_extra_confounds = False  # set True only for a deliberate teaching comparison
confounds = confounds_filename if use_extra_confounds else None

masker = NiftiLabelsMasker(
    labels_img=atlas_file,
    standardize="zscore_sample",
    memory="nilearn_cache",
    memory_level=2,
    verbose=1,
)

time_series = masker.fit_transform(nifti_filename_new, confounds=confounds)
print("time_series shape:", time_series.shape)


## Inspect the regional time series

A time-series plot helps participants understand that connectivity is computed from temporal covariation. The matrix in the next step is not computed from anatomy directly; it is computed from these regional signals.


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(time_series[:, :10])
plt.xlabel("Volume / scan number")
plt.ylabel("Standardized BOLD signal")
plt.title("First 10 atlas-region time series")
plt.tight_layout()


## Compute and display the ROI-to-ROI correlation matrix

The simplest static functional-connectivity estimate is Pearson correlation between pairs of regional time series. `ConnectivityMeasure(kind="correlation")` returns a square `regions × regions` matrix. Values near +1 indicate that two regions fluctuate together; values near -1 indicate anti-correlated fluctuations; values near 0 indicate little linear association.

This matrix is symmetric. The diagonal is 1 by definition because each region is perfectly correlated with itself.

For visualization, we set the diagonal to zero in a copy of the matrix so that the self-correlations do not dominate the color scale. Reordering the matrix can reveal block-like structure, but it can also make anatomical ordering less obvious.



In [ ]:
from nilearn.connectome import ConnectivityMeasure

correlation_measure = ConnectivityMeasure(kind="correlation")
correlation_matrix = correlation_measure.fit_transform([time_series])[0]

print("correlation_matrix shape:", correlation_matrix.shape)
print("range:", np.nanmin(correlation_matrix), "to", np.nanmax(correlation_matrix))

In [ ]:
correlation_matrix_for_plot = correlation_matrix.copy()
np.fill_diagonal(correlation_matrix_for_plot, 0)

plotting.plot_matrix(
    correlation_matrix_for_plot,
    figure=(10, 8),
    labels=labels,
    vmax=0.8,
    vmin=-0.8,
    reorder=True,
    title="Harvard-Oxford ROI-to-ROI correlation matrix",
)


## Probabilistic network maps: MSDL

Hard parcellations assign each voxel to one label. Probabilistic or spatial-map atlases instead define overlapping components or networks. The MSDL atlas is useful for teaching because it includes network labels and coordinates. Here we use `NiftiMapsMasker`, which is the appropriate masker for maps rather than discrete labels.


In [ ]:
msdl_atlas = datasets.fetch_atlas_msdl()
msdl_labels = msdl_atlas.labels
msdl_coords = msdl_atlas.region_coords

maps_masker = NiftiMapsMasker(
    maps_img=msdl_atlas.maps,
    standardize="zscore_sample",
    memory="nilearn_cache",
    memory_level=2,
    verbose=1,
)

msdl_time_series = maps_masker.fit_transform(nifti_filename_new)
msdl_correlation_matrix = correlation_measure.fit_transform([msdl_time_series])[0]
msdl_matrix_for_plot = msdl_correlation_matrix.copy()
np.fill_diagonal(msdl_matrix_for_plot, 0)

print("MSDL time_series shape:", msdl_time_series.shape)

## Visualize the MSDL probabilistic networks

MSDL allows to select from available networks, e.g. the Default-Mode Network or Language Networks

In [ ]:
# Extract only default mode network nodes
dmn_nodes = image.index_img(msdl_atlas.maps, [3, 4, 5, 6])
language_nodes = image.index_img(msdl_atlas.maps, [27,28,29,30,31])


# Plot MSDL probability atlas
plotting.plot_prob_atlas(dmn_nodes, cut_coords=(0, -60, 29), draw_cross=False,
                         annotate=False, title="DMN nodes in MSDL atlas")

# Plot MSDL probability atlas
plotting.plot_prob_atlas(language_nodes, cut_coords=(0, -60, 29), draw_cross=False,
                         annotate=False, title="Language nodes in MSDL atlas")

## Visualize the MSDL connectivity matrix and graph

This is the same operation as before, but with a different definition of nodes, here based on the MSDL atlas. 

In [ ]:
plotting.plot_matrix(
    msdl_matrix_for_plot,
    figure=(10, 8),
    labels=msdl_labels,
    vmax=0.8,
    vmin=-0.8,
    reorder=True,
    title="MSDL correlation matrix",
)

plotting.plot_connectome(
    msdl_matrix_for_plot,
    msdl_coords,
    edge_threshold="80%",
    colorbar=True,
    title="MSDL connectome, strongest edges",
)


## ROI-to-ROI Language-network example

The following uses a small set of canonical left-hemisphere language-related seed coordinates.

**Note:** A strong edge between language-related seeds does not prove that both sites are essential for language in an individual patient. It only indicates that the extracted BOLD time series were correlated in this run after the preprocessing choices applied above.

In [ ]:
language_seeds = {
    "L IFG / pars opercularis": (-48, 14, 18),
    "L posterior STG/STS": (-54, -42, 6),
    "L angular gyrus": (-42, -60, 36),
    "L middle temporal gyrus": (-58, -42, -8),
}

seed_labels = list(language_seeds.keys())
seed_coords = list(language_seeds.values())

seed_masker = NiftiSpheresMasker(
    seed_coords,
    radius=8,
    standardize="zscore_sample",
    verbose=1,
)

language_time_series = seed_masker.fit_transform(nifti_filename_new)
language_connectivity = correlation_measure.fit_transform([language_time_series])[0]
language_matrix_for_plot = language_connectivity.copy()
np.fill_diagonal(language_matrix_for_plot, 0)

plotting.plot_matrix(
    language_matrix_for_plot,
    labels=seed_labels,
    vmax=0.8,
    vmin=-0.8,
    title="Illustrative language-seed connectivity",
)

plotting.plot_connectome(
    language_matrix_for_plot,
    seed_coords,
    edge_threshold=0,
    colorbar=True,
    node_size=80,
    title="Illustrative language-seed connectome",
)


## Seed-to-Voxel Language-network example

The following uses a singe canonical left-hemisphere language-related seed coordinate and calculate the functional connectome between this and all other voxels .

**Note:** Again connectivity with the language seed does not prove that sites are essential for language in an individual patient. It only indicates that the extracted BOLD time series were correlated in this run after the preprocessing choices applied above.

In [ ]:
from nilearn.input_data import NiftiSpheresMasker

# Sphere radius in mm
sphere_radius = 8

# Sphere center in MNI-coordinate
sphere_coords = [(-48, 14, 18)]

seed_masker = NiftiSpheresMasker(sphere_coords, radius=sphere_radius, detrend=True,
                                 standardize=True, low_pass=0.1, high_pass=0.01,
                                 t_r=2.0, verbose=1, memory="nilearn_cache", memory_level=2)

In [ ]:

# Extract the signal from the regions
seed_time_series = seed_masker.fit_transform(nifti_filename_new, confounds=confounds_filename)
from nilearn.input_data import NiftiMasker

brain_masker = NiftiMasker(smoothing_fwhm=6, detrend=True, standardize=True,
                           low_pass=0.1, high_pass=0.01, t_r=2., verbose=1,
                           memory="nilearn_cache", memory_level=2)

brain_time_series = brain_masker.fit_transform(nifti_filename_new, confounds=confounds_filename)

seed_based_correlations = np.dot(brain_time_series.T, seed_time_series)
seed_based_correlations /= seed_time_series.shape[0]

## Plotting the seed-based correlation map¶


In [ ]:
seed_based_correlation_img = brain_masker.inverse_transform(seed_based_correlations.T)

conn_filename = Path(
    output_dir) / f"Seed-based_correlations_Broca.nii.gz"

seed_based_correlation_img.to_filename(conn_filename)

display = plotting.plot_stat_map(seed_based_correlation_img, threshold=0.8,
                                 cut_coords=sphere_coords[0])
display.add_markers(marker_coords=sphere_coords, marker_color='black',
                    marker_size=200)

In [ ]:
# print contrast
from ipywidgets import interact, interactive, fixed, interact_manual
from IPython.display import display
import ipywidgets as widgets
from ipyniivue import NiiVue, ShowRender, SliceType

# Create NiiVue instance with specific settings
nv = NiiVue(
    loading_text="waiting",
    back_color=(1, 1, 1, 1),
    show_3d_crosshair=True,
    is_colorbar=True,
    multiplanar_show_render=ShowRender.ALWAYS,
)

# Set initial configuration
nv.set_radiological_convention(False)
nv.set_slice_type(SliceType.MULTIPLANAR)
nv.set_slice_mm(False)
nv.set_interpolation(True)

# Load 4D volume with paired HEAD and BRIK files
nv.load_volumes(
    [
        {
            "path": "./ILAE_NeuroimagingSchool/dataset/anat/sub-ILAEDemo001_ses-01_run-01_space-MNI_desc-preproc_T1w.nii.gz",
        },
        {
            "path": "./ILAE_NeuroimagingSchool/outputs/Seed-based_correlations_Broca.nii.gz",
            "colormap": "warm",
            "colormap_negative": "winter",
            "cal_min": 3,
            "cal_max": 6,
            "cal_min_neg": -6,
            "cal_max_neg": -3,
        },
    ]
)


# Hide colorbar for anatomical scan
nv.volumes[0].colorbar_visible = False

# Set initial overlay outline
nv.overlay_outline_width = 0.25

# High DPI checkbox
dpi_checkbox = widgets.Checkbox(
    value=True,
    description="High DPI",
    tooltip="Higher resolution for 'retina' displays",
)

# Negative colors checkbox
negative_checkbox = widgets.Checkbox(value=True, description="Negative Colors")

# Smooth checkbox
smooth_checkbox = widgets.Checkbox(
    value=False,
    description="Smooth",
    tooltip=(
        "Trilinear interpolation blurs data, "
        "but can change which voxels survive a threshold"
    ),
)

# World space checkbox
world_checkbox = widgets.Checkbox(value=False, description="World Space")


# Outline width slider
outline_slider = widgets.IntSlider(
    min=0, max=4, value=1, description="Outline", continuous_update=True, readout=False
)

# Alpha mode dropdown
alpha_dropdown = widgets.Dropdown(
    options=[
        ("Restrict colorbar to range", 0),
        ("Colorbar from 0, transparent subthreshold", 1),
        ("Colorbar from 0, translucent subthreshold", 2),
    ],
    value=0,
    description="Alpha Mode:",
)

# threshold sliders

pos_stat_range = widgets.IntRangeSlider(
    value=[0.8,1],
    min=1,
    max=10,
    step=1,
    description='+Threshold:',
    continuous_update=True,
    readout=False,
)

neg_stat_range = widgets.IntRangeSlider(
    value=[0.8, 1],
    min=1,
    max=10,
    step=1,
    description='-Threshold:',
    continuous_update=True,
    readout=False,
)
# Location display
location_output = widgets.HTML(value="&nbsp;")

## Setup Event Handlers

def on_dpi_change(change):
    """Handle DPI checkbox changes."""
    nv.set_high_resolution_capable(change["new"])

def on_negative_change(change):
    """Handle negative colormap checkbox changes."""
    neg_stat_range.disabled = not change["new"]
    if change["new"]:
        nv.set_colormap_negative(nv.volumes[1].id, "winter")
    else:
        nv.set_colormap_negative(nv.volumes[1].id, "")

def on_smooth_change(change):
    """Handle smooth interpolation checkbox changes."""
    nv.set_interpolation(not change["new"])


def on_world_change(change):
    """Handle world space checkbox changes."""
    nv.set_slice_mm(change["new"])


def on_outline_change(change):
    """Handle outline width slider changes."""
    nv.overlay_outline_width = 0.25 * change["new"]


def on_alpha_mode_change(change):
    """Handle alpha mode dropdown changes."""
    nv.volumes[1].colormap_type = change["new"]

def on_pos_stat_change(change):
    """Set threshold for statistical overlay."""
    low, high = change["new"]
    nv.volumes[1].cal_min = low
    nv.volumes[1].cal_max = high 

def on_neg_stat_change(change):
    """Set threshold for statistical overlay."""
    low, high = change["new"]
    nv.volumes[1].cal_min_neg = -low
    nv.volumes[1].cal_max_neg = -high 

@nv.on_location_change
def handle_location_change(data):
    """Update location display when crosshair moves."""
    location_output.value = f"&nbsp;&nbsp;{data['string']}"

# Attach event handlers
dpi_checkbox.observe(on_dpi_change, names="value")
negative_checkbox.observe(on_negative_change, names="value")
smooth_checkbox.observe(on_smooth_change, names="value")
world_checkbox.observe(on_world_change, names="value")
outline_slider.observe(on_outline_change, names="value")
alpha_dropdown.observe(on_alpha_mode_change, names="value")

# Initialize values
on_outline_change({"new": outline_slider.value})
on_alpha_mode_change({"new": alpha_dropdown.value})

pos_stat_range.observe(on_pos_stat_change, names="value")
neg_stat_range.observe(on_neg_stat_change, names="value")
## Display All

# Organize controls
controls_row1 = widgets.HBox(
    [dpi_checkbox, negative_checkbox, smooth_checkbox, world_checkbox]
)
controls_row2 = widgets.HBox([neg_stat_range, pos_stat_range])
controls_row3 = widgets.HBox([outline_slider, alpha_dropdown])

# Create main layout
controls = widgets.VBox(
    [controls_row1, controls_row2, controls_row3, location_output]
)

# Display everything
display(widgets.VBox([controls, nv]))
